# 02 · Local risk score: rule baseline vs learned model
The shipped scorer is a transparent weighted score (`detection/scorer.py`). Here we check whether a learned model (LightGBM if installed, else gradient boosting from scikit-learn) beats it, using a **ring-grouped split** so accounts from one ring never appear in both train and test.

⚠ Labels come from the simulator, so AUCs are optimistic. The point is relative comparison and calibration shape.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "backend"))   # run from repo root or notebooks/
import pandas as pd
import matplotlib.pyplot as plt
from app.simulator import generate_dataset
from app.detection.features import FEATURE_NAMES, compute_features_batch
from app.detection.scorer import RiskScorer
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.calibration import calibration_curve
try:
    from lightgbm import LGBMClassifier as GBM
    MODEL = "LightGBM"
    mk = lambda: GBM(n_estimators=150, learning_rate=0.05, num_leaves=15, min_child_samples=10, verbose=-1, random_state=0)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier as GBM
    MODEL = "sklearn HistGradientBoosting"
    mk = lambda: GBM(max_iter=150, learning_rate=0.05, random_state=0)
print("model:", MODEL)

In [ ]:
data = generate_dataset(seed=42)
txs = data["transactions"]
ring_of = {a: i for i, m in enumerate(data["ring_meta"]) for a in m["members"]}
feats = compute_features_batch(txs)
accts = sorted(feats)
X = np.array([[feats[a][n] for n in FEATURE_NAMES] for a in accts])
y = np.array([a in ring_of for a in accts], dtype=int)
groups = np.array([ring_of.get(a, 10_000 + i) for i, a in enumerate(accts)])
tr, te = next(GroupShuffleSplit(test_size=0.3, random_state=0).split(X, y, groups))
print(X.shape, "positives:", y.sum(), "| train/test:", len(tr), len(te))

In [ ]:
scorer = RiskScorer()
rule = np.array([scorer.score(feats[a]).risk_score for a in accts])
model = mk().fit(X[tr], y[tr])
ml = model.predict_proba(X[te])[:, 1]
res = pd.DataFrame({
    "model": ["rule scorer", MODEL],
    "ROC-AUC": [roc_auc_score(y[te], rule[te]), roc_auc_score(y[te], ml)],
    "PR-AUC":  [average_precision_score(y[te], rule[te]), average_precision_score(y[te], ml)],
})
res

## Calibration — is a score of 0.8 really ~80% ring?

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for name, p in (("rule", rule[te]), (MODEL, ml)):
    frac, mean = calibration_curve(y[te], p, n_bins=8, strategy="quantile")
    ax.plot(mean, frac, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("predicted"); ax.set_ylabel("observed ring rate"); ax.legend(); plt.show()

## Feature importance (learned model)

In [ ]:
imp = pd.Series(getattr(model, "feature_importances_", np.zeros(len(FEATURE_NAMES))), index=FEATURE_NAMES).sort_values()
imp.plot.barh(figsize=(5, 3)); plt.show()

**Decision:** keep the transparent scorer for the demo (every alert needs human-readable reasons). If the learned model is clearly better, use it for ranking and keep the rule reasons for explanation.